# Initializing the environment

In [ ]:
from gymnasium import make
from samples.llm_interface import OGPT4Interfacer
import os
import random
os.environ["OPENAI_API_KEY"] = ""

MAX_EPISODES = 5

# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=[('high_elf_mage.yml','Joe'), ('halfling_rogue.yml', 'Roger')],
    enemies=[('high_elf_fighter.yml', 'Mike'), ('halfling_rogue.yml','Spencer')],
    )
observation, info = env.reset(seed=42)




agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agents[character.name] = (OGPT4Interfacer(debug=True, explain=True, name=character.name), gr, character)


/home/thomas/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/samples/llm_interface.py:659: SyntaxWarning: invalid escape sequence '\d'
  regex = "\d"


Joe rolled initiative d20(18) + 2 value 20.15
Roger rolled initiative d20(3) + 5 value 8.2
Mike rolled initiative d20(19) + 5 value 24.2
Spencer rolled initiative d20(14) + 5 value 19.2
Joe rolled initiative d20(2) + 2 value 4.15
Roger rolled initiative d20(1) + 5 lucky -> d20(3) + 5 value 8.2
Mike rolled initiative d20(7) + 5 value 12.2
Spencer rolled initiative d20(8) + 5 value 13.2
Combat begins with 4 players.
Players: <p>Joe (wizard-2) Team a</p>
<p>Roger (rogue-2) Team a</p>
<p>Mike (fighter-2) Team b</p>
<p>Spencer (rogue-2) Team b</p>
======== Spencer starts their turn. ========
Spencer attacked Joe with Shortbow and hits with attack roll d20(11) + 7 = 18.
Joe took d6(1) + 5 = 6 piercing damage. 
Spencer moved to [4, 9] 5 feet
Spencer moved to [5, 8] 5 feet
Spencer moved to [6, 7] 5 feet
Spencer moved to [7, 6] 5 feet
Spencer moved to [8, 5] 5 feet
======== Roger starts their turn. ========


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


In [2]:
from natural20.map_renderer import MapRenderer
map_r = MapRenderer(env.env.env.battle.maps[0])
print(map_r.render())

···P················
····················
·······G············
········R···········
····················
········R···········
····················
····················
····················
····················
····················
····················
····················
····················
····················



In [3]:
env.env.env.players

[('a', 'H', Joe, [3, 0]),
 ('a', 'H', Roger, [8, 3]),
 ('b', 'E', Mike, [7, 2]),
 ('b', 'E', Spencer, [3, 10])]

In [ ]:
{
    "Hihg_elf_mage" : "P",
    "fighter" : "G",
    "rogue" : "R",
}

In [17]:
env.env.env.battle.maps[0]??

Type:        Map
String form: <natural20.map.Map object at 0x7fa9cbded1d0>
File:        ~/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/natural20/map.py
Source:     
class Map():
    def __init__(self, session, map_file_path, name=None, properties=None, skip_setup=False):
        self.name = name
        self.session = session
        self.terrain = {}
        self.spawn_points = {}
        self.area_triggers = {}
        self.map = []
        if properties:
            self.properties = properties
        else:
            self.properties = self.load(map_file_path)
        base = self.properties.get('map', {}).get('base', [])

        self.size = [len(base[0]), len(base)]
        # print(f"map size: {self.size}")
        self.feet_per_grid = self.properties.get('grid_size', 5)
        self.base_map = []
        self.base_map_1 = []
        self.base_map_2 = []
        self.objects = []
        self.tokens = []
        self.unaware_npcs = []
        self.entities = {}  # A

In [17]:
agents

{'Spencer': (<samples.llm_interface.OGPT4Interfacer at 0x7f1fef3afb70>,
  'b',
  Spencer),
 'Mike': (<samples.llm_interface.OGPT4Interfacer at 0x7f1fef2154f0>,
  'b',
  Mike),
 'Roger': (<samples.llm_interface.OGPT4Interfacer at 0x7f1fef2178b0>,
  'a',
  Roger),
 'Joe': (<samples.llm_interface.OGPT4Interfacer at 0x7f1fef217960>, 'a', Joe)}

In [4]:
env.env.env.battle.current_turn()

Roger

In [9]:
info["entity_mappings"]

{'cleric-3': 1,
 'wizard-2': 2,
 'rogue-2': 3,
 'fighter-2': 4,
 'owl': 5,
 'animated broom': 6,
 'specter': 7,
 'rokvakgor': 8,
 'bat': 9,
 'dargragrog': 10,
 'thakdukrog': 11,
 'animated armor': 12,
 'tormog': 13,
 'human guard': 14,
 'owlbear': 15,
 'wolf': 16,
 'skeleton': 17}

In [3]:
print(env.render())

···········_
····?······_
·?····?····_
···········_
···········_
···········_
······A····_
···········_
···········_
···········_
···········_
···········_


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:317: UserWarning: WARN: No render modes was declared in the environment (env.metadata['render_modes'] is None or not defined), you may have trouble when calling `.render()`.
  logger.warn(


In [ ]:
def update_all_agents(agents_name, sender, content):
    for name in agents_name:
        agents[name][0].register_conversation(sender, content)

def initiate_conversation(agents_name):
    for name in agents_name:
        agents[name][0].initiate_conversation(sender, content)

def close_conversation(agents_name):
    for name in agents_name:
        agents[name][0].close_conversation(self)

def run_conversation(sender, content):
    sender_gr = agents[sender][1]
    agent_in_the_conv = []
    for name, (_, gr, _) in agents.items():
        if sender_gr == gr and name != sender:
            agent_in_the_conv.append(name)
    agent_in_the_conv.append(sender)
    initiate_conversation(agent_in_the_conv)
    update_all_agents(agent_in_the_conv, sender, content)
    conv_alive = True
    conv_step = 0
    while conv_alive:
        conv_step += 1
        conv_alive = False
        for name in agent_in_the_conv:
            action, content = agents[name][0].select_action_for_state(observation, info, is_conversation=True)
            if action == -2:
                update_all_agents(agent_in_the_conv, name, content)
                conv_alive = True
            elif action != -3:
                raise ValueError(f"A non conversation action {action} was used during a conversation by agent {name}")
    close_conversation(agent_in_the_conv)
    return conv_step


In [6]:
# Select an action based on the initial state
current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]

action = current_agent.select_action_for_state(observation, info)
print(f"Selected action: {action}")
# terminal = False
# episode = 0
# while not terminal and episode < MAX_EPISODES:
#     episode += 1
#     observation, reward, terminal, truncated, info = env.step(action)
#     if not terminal and not truncated:
#         print(env.render())
#         current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]
#         action = current_agent.select_action_for_state(observation, info)
#         print(f"Selected action: {action}")

#     if terminal or truncated:
#         print(f"Reward: {reward}")
#         break
# action

prompt: -------------------------------
It is currently your turn
Your hero character (a level 2 rogue) is denoted by P. And you have an enemy donoted by E (a level 2 fighter)which you must defeat. 
Your health is at [100.]% specifically 16/16 
Your Enemies health is at [100.]%
Your current conditions are:
Your enemies current conditions are:
You have the following available actions and movement available:

Available movement: [25]ft
Available actions: 1
Bonus actions: 1
Reactions: 1



Here is a rough sketch of the map that considers line of sight to the enemy.
Here is the map:
____________
____________
____________
.A..........
............
.....E......
......P.....
............
......E.....
............
............
............
areas with no characters are represented by a dot (.)
the hero character is represented by a (P)
the enemy character is represented by an (E)
Allies or Party Members are represented by an (A)
Neutral characters are represented by a question mark (?)
areas ou

KeyError: 'action'

In [25]:
observation.keys()

dict_keys(['map', 'turn_info', 'conditions', 'health_pct', 'player_equipped', 'health_enemy', 'enemy_conditions', 'enemy_reactions', 'player_ac', 'enemy_ac', 'ability_info', 'player_type', 'enemy_type', 'spell_slots', 'movement', 'is_reaction'])

In [8]:
observation["health_enemy"]

array([1.])